In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import Descriptors
from tqdm.auto import tqdm
import mols2grid

In [ ]:
tqdm.pandas()

In [ ]:
def butina_cluster(mol_list, cutoff=0.35):
    # Create Morgan fingerprint generator with radius 3 and 2048 bits
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
    # Generate fingerprints for all molecules
    fp_list = [generator.GetFingerprint(m) for m in mol_list]
    dists = []
    nfps = len(fp_list)
    for i in range(1, nfps):
        sims = DataStructs.BulkTanimotoSimilarity(fp_list[i], fp_list[:i])
        dists.extend([1-x for x in sims])
    mol_clusters = Butina.ClusterData(dists, nfps, cutoff, isDistData=True)
    cluster_id_list = [0]*nfps
    for idx, cluster in enumerate(mol_clusters, 1):
        for member in cluster:
            cluster_id_list[member] = idx
    return cluster_id_list

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/PatWalters/practical_cheminformatics_tutorials/main/data/dude_erk2_mk01.csv")

In [ ]:
df.head()

In [ ]:
mols2grid.display(df)

In [ ]:
df['mol'] = df.SMILES.progress_apply(Chem.MolFromSmiles)

In [ ]:
%time df['Cluster'] = butina_cluster(df.mol.values)

In [ ]:
mols2grid.display(df,subset=["img","ID","Cluster"])

In [ ]:
df["logP"] = df.mol.progress_apply(Descriptors.MolLogP)

In [ ]:
mols2grid.display(df,subset=["img","ID","Cluster","logP"],transform={"logP": lambda x: f"{x:.2f}"})

In [ ]:
df.sort_values(["Cluster","logP"],inplace=True)

In [ ]:
mols2grid.display(df,subset=["img","ID","Cluster","logP"],transform={"logP": lambda x: f"{x:.2f}"},prerender=True)

In [ ]:
df_unique = df.drop_duplicates("Cluster")

In [ ]:
mols2grid.display(df_unique,subset=["img","ID","Cluster","logP"],transform={"logP": lambda x: f"{x:.2f}"},prerender=True)

In [ ]:
grid = mols2grid.display(
    df_unique,
    subset=["img", "ID", "Cluster", "logP"],
    transform={"logP": lambda x: f"{x:.2f}"},
    prerender=True
)

grid